In [0]:
%python

# Batch 2 — modify 500 customers from batch 1
# Pick 500 random customer_unique_ids
# Change their city to a different Brazilian city
# Change their state accordingly
# Keep everything else the same

# Example change:
# Before: customer_unique_id=abc123, city=sao paulo, state=SP
# After:  customer_unique_id=abc123, city=rio de janeiro, state=RJ

In [0]:
%python

%run /Workspace/Users/motloungmiya@gmail.com/medallion-ecommerce-scd/config/pipeline_config.py

# Config imported ✓

from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.read.csv(SOURCE_FILE, header=True, inferSchema=True)
df = df.withColumn("row_id", F.row_number().over(Window.orderBy("customer_id")))

# ── Helper: simulate location change ─────────────────────────
def change_location(df, new_city, new_state):
    return df \
        .withColumn("customer_city",  F.lit(new_city)) \
        .withColumn("customer_state", F.lit(new_state))

# ── BATCH 1: Initial load — all customers, all Jan ───────────
batch1 = df.withColumn(
    "modified_date", F.lit("2024-01-01").cast("date")
)
batch1.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/batch_1")
print(f"Batch 1: {batch1.count():,} rows written")

# ── BATCH 2: 500 customers move to Curitiba/PR ───────────────
moved_500 = df.orderBy(F.rand(seed=42)).limit(500)

moved_500_updated = change_location(moved_500, "curitiba", "PR") \
    .withColumn("modified_date", F.lit("2024-02-01").cast("date"))

unchanged_b2 = df.join(
    moved_500.select("customer_id"), on="customer_id", how="left_anti"
).withColumn("modified_date", F.lit("2024-01-01").cast("date"))  # ← stays Jan

batch2 = unchanged_b2.unionByName(moved_500_updated)
batch2.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/batch_2")
print(f"Batch 2: {batch2.count():,} rows ({moved_500_updated.count()} changed)")

# ── BATCH 3: 300 more customers move to Salvador/BA ──────────
moved_300 = df.orderBy(F.rand(seed=999)).limit(300)

moved_300_updated = change_location(moved_300, "salvador", "BA") \
    .withColumn("modified_date", F.lit("2024-03-01").cast("date"))

# Unchanged = not in either moved set
all_moved_ids = moved_500.select("customer_id") \
    .unionByName(moved_300.select("customer_id"))

unchanged_b3 = df.join(
    all_moved_ids, on="customer_id", how="left_anti"
).withColumn("modified_date", F.lit("2024-01-01").cast("date"))

batch3 = unchanged_b3 \
    .unionByName(moved_500_updated) \
    .unionByName(moved_300_updated)

batch3.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/batch_3")
print(f"Batch 3: {batch3.count():,} rows ({moved_300_updated.count()} new changes)")

In [0]:
%python

for batch, path in [("batch_1","batch_1"), ("batch_2","batch_2"), ("batch_3","batch_3")]:
    df_b = spark.read.format("delta").load(f"{BRONZE_PATH}/{path}")
    print(f"\n{batch}:")
    df_b.groupBy("modified_date").count().orderBy("modified_date").show()